In [ ]:
from pyspark.sql import functions as F
from infra.spark_utils import build_spark, normalize_ptbr_number
from infra.tickers_cache import get_tickers_price, update_tickers_cache

In [ ]:
import sys
print(sys.executable)

In [ ]:
spark = build_spark()

In [ ]:
import sys
import os

print("Notebook Python:", sys.executable)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_DRIVER_PYTHON:", os.environ.get("PYSPARK_DRIVER_PYTHON"))

In [ ]:
spark = build_spark()
file_path = "../data/bronze/forms/economias.csv"
df = spark.read.csv(
    path = file_path,
    sep = ",", 
    header=True,
    multiLine=True # se n tiver quebra a logica 
)
df

In [ ]:
df = (df.withColumn(
    'resumo',
    F.regexp_replace(F.col('resumo'), r"\r\n|\r", "\n")
    )
    .withColumn(
    'resumo',
    F.split(F.col('resumo'), '\n')
    )
    .withColumn(
            'resumo',
            F.explode(F.col('resumo'))
        )
)
df.show()

In [ ]:
parts = F.split(F.col('resumo'), r"\|")
df = (df
      .withColumn("tipo", F.trim(parts.getItem(0)))
      .withColumn("nome", F.trim(parts.getItem(1)))
      .withColumn("qtd", F.trim(parts.getItem(2)))
      .withColumn("preco_medio", F.trim(parts.getItem(3)))
      .withColumn("preco_atual", F.trim(parts.getItem(4)))
      .withColumn("tipo", normalize_ptbr_number(F.col("tipo")))
      .withColumn("nome", normalize_ptbr_number(F.col("nome")))
      .withColumn("qtd", normalize_ptbr_number(F.col("qtd")))
      .withColumn("preco_medio", normalize_ptbr_number(F.col("preco_medio")))
      .withColumn("preco_atual", normalize_ptbr_number(F.col("preco_atual")))
    )

df.show()

In [ ]:
mapping_month = F.create_map(
    F.lit(1), F.lit("Jan"),  
    F.lit(2), F.lit("Fev"),  
    F.lit(3), F.lit("Mar"),  
    F.lit(4), F.lit("Abr"),  
    F.lit(5), F.lit("Mai"),  
    F.lit(6), F.lit("Jun"),  
    F.lit(7), F.lit("Jul"),  
    F.lit(8), F.lit("Ago"),  
    F.lit(9), F.lit("Set"),  
    F.lit(10), F.lit("Out"), 
    F.lit(11), F.lit("Nov"),
    F.lit(12), F.lit("Dec")
)

In [ ]:
df = (df
      .withColumn(
          'data_apuracao',
          F.to_date(F.col('data_apuracao'), format='dd/MM/yyyy')
      )
      .withColumn(
          'ano',
          F.year(F.col('data_apuracao'))
      )
      .withColumn(
          'ano',
          F.year(F.col('data_apuracao'))
      )
      .withColumn(
          'mes_num',
          F.month(F.col('data_apuracao'))
      )
      .withColumn(
          'mes',
          mapping_month[F.col('mes_num')]
      )
)

df.show()


In [ ]:
tickers_internacionais = ['BERK34', 'IVVB11', 'AAPL34', 'MSFT34', 'NVDC34', 'GOGL34', 'AMZO34', 'TSLA34', 'META34', 'MELI34']

df = (df
      .withColumn(
          'exposicao', 
          F.when(F.col('nome').isin(tickers_internacionais), "internacional")
          .otherwise("nacional")
      )
      .withColumn(
          'tipo',
          F.lower(F.col('tipo'))
      )
      .withColumn(
          'instituicao_fin',
          F.upper(F.col('instituicao_fin'))
      )
      .withColumn(
          'tipo',
          F.lower(F.col('tipo'))
      )
      .withColumn(
          'nome',
          F.when(F.col('tipo') == 'stock', F.upper(F.col('nome')))
          .otherwise(F.lower(F.col('nome')))
      )
)

df.show()

In [ ]:
df = (df.select(
    F.to_timestamp(F.col('timestamp'), "dd/MM/yyyy HH:mm:ss").alias('timestamp'),
    F.to_date(F.col('data_apuracao'), "dd/MM/yyyy").alias('data_apuracao'),
    F.col('ano').cast('int').alias('ano'),
    F.col('mes_num').cast('int').alias('mes_num'),
    F.col('mes').cast('string').alias('mes'),
    F.col('instituicao_fin').cast('string').alias('instituicao_fin'),
    F.col('resumo').cast('string').alias('resumo'),
    F.col('tipo').cast('string').alias('tipo'),
    F.col('nome').cast('string').alias('nome'),
    F.col('qtd').cast('double').alias('qtd'),
    F.col('preco_medio').cast('double').alias('preco_medio'),
    F.col('preco_atual').cast('double').alias('preco_atual'),
    F.col('aporte').cast('double').alias('aporte'),
    F.col('exposicao').cast('string').alias('exposicao')
    )
)
df.show()

In [ ]:
df_test = get_tickers_price(df)
df_test.show()

In [ ]:
test_df = spark.createDataFrame(
    [("BERK34", 120.5), ("BOVA11", None)],
    ["ticker", "close"]
)

test_df.show()

In [ ]:
spark.stop()